In [97]:
import os

from smolagents import ToolCollection, OpenAIServerModel, ToolCallingAgent, PromptTemplates, FinalAnswerPromptTemplate
from mcp import StdioServerParameters

In [98]:
os.environ['NEBIUS_API_KEY'] = open('secret.txt', 'r').read().strip()

In [1]:
[None] * 3

[None, None, None]

In [16]:
server = StdioServerParameters(
    command="docker",
    args=["exec", "-i", "youthful_ride", "node", "/app/build/index.js"]
)

In [17]:
MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"

model = OpenAIServerModel(
    model_id=MODEL,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

In [18]:
with ToolCollection.from_mcp(
    server_parameters=server,
    trust_remote_code=True,
    structured_output=False
) as tools:
    print([t.name for t in tools.tools])  # посмотреть, какие инструменты есть

['search_proteins', 'get_protein_info', 'search_by_gene', 'get_protein_sequence', 'get_protein_features', 'compare_proteins', 'get_protein_homologs', 'get_protein_orthologs', 'get_phylogenetic_info', 'get_protein_structure', 'get_protein_domains_detailed', 'get_protein_variants', 'analyze_sequence_composition', 'get_protein_pathways', 'get_protein_interactions', 'search_by_function', 'search_by_localization', 'batch_protein_lookup', 'advanced_search', 'search_by_taxonomy', 'get_external_references', 'get_literature_references', 'get_annotation_confidence', 'export_protein_data', 'validate_accession', 'get_taxonomy_info']


In [90]:
SYSTEM_PROMPT = """
You are a bioinformatics research agent connected exclusively to the Augmented-Nature-UniProt-MCP-Server.

Your task is to retrieve precise protein information from UniProt and its linked databases for a given gene or protein name.

Scope of retrieval:
1. Canonical UniProt entry (include a FASTA link, not the raw sequence).
2. Isoforms (alternative splicing) — names, UniProt IDs, and lengths.
3. Functional regions and sequence features — domains, motifs, regions, sites, mutagenesis.
4. Natural variants and mutagenesis experiments with described functional effects.
5. Post-translational modifications (PTMs) such as phosphorylation, acetylation, methylation, ubiquitination, etc.
6. Subcellular location annotations.
7. Cross-references: HGNC, Ensembl, PDB, Reactome, Gene Ontology (GO), KEGG.

Rules:
- Use **only** data obtained directly from MCP tools (e.g., `search_proteins`, `get_protein_info`, `get_protein_features`, `get_cross_references`).
- **Do not guess or fabricate** any facts not explicitly found in tool outputs.
- Keep biological names and identifiers exactly as in UniProt.
- Prefer full lists if multiple values exist.
"""

In [111]:
gene_or_protein_name = "APOE"

user_prompt = f"""
Protein: {gene_or_protein_name}

Retrieve all available UniProt data per the scope above, using only MCP tool outputs. 
Do not invent or infer missing information.
Search recommendations: Use more than 1 search results. String indices must be integers, not 'str'.
Rules and priorities:
1. **Always prefer reviewed (Swiss-Prot) entries** over unreviewed (TrEMBL) ones when both exist.
2. If the search results include multiple entries, select the one explicitly marked as “UniProtKB reviewed (Swiss-Prot)” — this is the canonical human record.
3. Never claim that data is “missing” if a reviewed canonical entry was found.
4. If both a fragment and a full canonical entry are present, use the full reviewed one and ignore the fragment.
5. Use only the information retrieved from MCP tool outputs. Do not add facts from external knowledge.
6. If no reviewed entry is found at all, then and only then mention that data may be incomplete.

When writing the summary:
- Use bullet points or short factual sentences under each section.
- Include FASTA links and UniProt IDs where available.
- Do not include commentary about the search process or tool execution.
- Do not add speculative language like “may be”, “possibly”, “appears to”.
- All information must be based ONLY on the data retrieved by via tools. 
If any of the sections are unavailable print "Info unavailable" - do not invent facts.

Write a concise, readable plain-text summary of the retrieved UniProt data.
Stick to this structure in output:
1. **Canonical sequence and isoforms**
   - Retrieve the canonical FASTA sequence and all annotated isoforms (alternative splicing variants).  
   - Note sequence length and UniProt accession for each.

2. **Functional sequence intervals**
   - From `features`: list all annotated *regions, domains, motifs, sites,* and *mutagenesis* entries.  
   - For each, include:
     • name of interval  
     • amino-acid positions  
     • description or known binding partner  
     • experimental notes (e.g., “loss of KEAP1 binding”, “increased transcriptional activity”).

3. **Natural variants and mutagenesis data**
   - Extract annotated variants and experimental mutagenesis results.  
   - For each, specify: position → substitution → observed functional consequence (e.g. “S40A — abolishes phosphorylation”).  
   - Focus on those affecting activity, binding, or stability.

4. **Post-translational modifications (PTM)**
   - Summarize all PTM annotations (phosphorylation, acetylation, ubiquitination, etc.) with residue positions and known effect on function.

5. **Cross-references**
   - Include available cross-links to HGNC, Ensembl, PDB, Reactome, Gene Ontology, and KEGG.  
   - Briefly describe what each reference contributes (e.g. “Reactome → NRF2 pathway in oxidative stress response”).

6. **Subcellular location**
   - State primary and secondary locations from UniProt annotation (e.g. “Cytoplasm → Nucleus translocation upon activation”).  
   - Note relevance to function or degradation (e.g. “KEAP1-mediated cytoplasmic retention”).

7. **Longevity relevance**
   - When possible, highlight how these functional regions or variants are implicated in lifespan regulation (e.g. “Neoaves KEAP1 mutation → constitutive NRF2 activation → increased stress resistance”).

When calling MCP tools, use only the UniProt tools available in the current MCP server (e.g., `bc_search_uniprot_entries`, `bc_get_uniprot_entry`, `bc_get_uniprot_features`, `bc_get_uniprot_variants`, `bc_get_uniprot_crossrefs`, `bc_get_uniprot_subcellular_location`, etc. — depending on implementation).

Return concise, human-readable research text suitable for direct inclusion into a WikiCrow-style article.  
Do **not** output JSON or code blocks — only clean text with section headers.
"""

In [112]:
with ToolCollection.from_mcp(
    server_parameters=server,
    trust_remote_code=True,
    structured_output=False
) as tools:
    agent = ToolCallingAgent(
        tools=[*tools.tools],
        add_base_tools=False,
        model=model,
        max_steps=1,
    )
    agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
    # agent.prompt_templates["final_answer"] = FINAL_RESULT_PROMPT

    # try:
    out = agent.run(user_prompt)
        # print(out)
    # except Exception as e:
    #     print(f"ERROR: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Protein: APOE                                                                                                   │
│                                                                                                                 │
│ Retrieve all available UniProt data per the scope above, using only MCP tool outputs.                           │
│ Do not invent or infer missing information.                                                                     │
│ Search recommendations: Use more than 1 search results. String indices must be integers, not 'str'.             │
│ Rules and priorities:                                                                                           │
│ 1. **Always prefer reviewed (Swiss-Prot) entries** over unreviewed (TrEMBL) ones when both exist.               │
│ 2. If the search results include multiple entries, select the one explicitly marked as “UniProtKB reviewed      │
│ (Swiss-Prot)” — this is the canonical human record.                                                             │
│ 3. Never claim that data is “missing” if a reviewed canonical entry was found.                                  │
│ 4. If both a fragment and a full canonical entry are present, use the full reviewed one and ignore the          │
│ fragment.                                                                                                       │
│ 5. Use only the information retrieved from MCP tool outputs. Do not add facts from external knowledge.          │
│ 6. If no reviewed entry is found at all, then and only then mention that data may be incomplete.                │
│                                                                                                                 │
│ When writing the summary:                                                                                       │
│ - Use bullet points or short factual sentences under each section.                                              │
│ - Include FASTA links and UniProt IDs where available.                                                          │
│ - Do not include commentary about the search process or tool execution.                                         │
│ - Do not add speculative language like “may be”, “possibly”, “appears to”.                                      │
│ - All information must be based ONLY on the data retrieved by via tools.                                        │
│ If any of the sections are unavailable print "Info unavailable" - do not invent facts.                          │
│                                                                                                                 │
│ Write a concise, readable plain-text summary of the retrieved UniProt data.                                     │
│ Stick to this structure in output:                                                                              │
│ 1. **Canonical sequence and isoforms**                                                                          │
│    - Retrieve the canonical FASTA sequence and all annotated isoforms (alternative splicing variants).          │
│    - Note sequence length and UniProt accession for each.                                                       │
│                                                                                                                 │
│ 2. **Functional sequence intervals**                                                                            │
│    - From `features`: list all annotated *regions, domains, motifs, sites,* and *mutagenesis* entries.          │
│    - For each, include:                                                                                         │
│      • name of interval                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_by_gene' with arguments: {'gene': 'APOE', 'organism': 'Homo sapiens', 'size': 5}          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "results": |
    {
      "entryType": "UniProtKB reviewed (Swiss-Prot)",
      "primaryAccession": "P02649",
      "secondaryAccessions": |
        "B2RC15",
        "C0JYY5",
        "Q9P2S4"
      ],
      "uniProtkbId": "APOE_HUMAN",
      "entryAudit": {
        "firstPublicDate": "1986-07-21",
        "lastAnnotationUpdateDate": "2025-06-18",
        "lastSequenceUpdateDate": "1986-07-21",
        "entryVersion": 277,
        "sequenceVersion": 1
      },
      "annotationScore": 5,
      "organism": {
        "scientificName": "Homo sapiens",
        "commonName": "Human",
        "taxonId": 9606,
        "lineage": |
          "Eukaryota",
          "Metazoa",
          "Chordata",
          "Craniata",
          "Vertebrata",
          "Euteleostomi",
          "Mammalia",
          "Eutheria",
          "Euarchontoglires",
          "Primates",
          "Haplorrhini",
          "Catarrhini",
          "Hominidae",
          "Homo"
        ]
      },
      "proteinExistence": "1: Evidence at protein level",
      "proteinDescription": {
        "recommendedName": {
          "fullName": {
            "evidences": |
              {
                "evidenceCode": "ECO:0000305"
              }
            ],
            "value": "Apolipoprotein E"
          },
          "shortNames": |
            {
              "value": "Apo-E"
            }
          ]
        },
        "flag": "Precursor"
      },
      "genes": |
        {
          "geneName": {
            "evidences": |
              {
                "evidenceCode": "ECO:0000312",
                "source": "HGNC",
                "id": "HGNC:613"
              }
            ],
            "value": "APOE"
          }
        }
      ],
      "comments": |
        {
          "texts": |
            {
              "evidences": |
                {
                  "evidenceCode": "ECO:0000250",
                  "source": "UniProtKB",
                  "id": "P08226"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "12950167"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "14754908"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "1530612"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "1911868"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "1917954"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "20030366"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "20303980"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "2063194"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "23620513"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "23676495"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "2762297"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "28111074"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source":

[Step 1: Duration 6.97 seconds| Input tokens: 4,276 | Output tokens: 37]

Reached max steps.

[Step 2: Duration 42.41 seconds| Input tokens: 132,638 | Output tokens: 1,587]

In [113]:
print(out)

1. **Canonical sequence and isoforms**  
   - Canonical sequence: UniProt accession P02649, length 317 amino acids.  
   - FASTA link: [https://www.uniprot.org/uniprotkb/P02649/entry](https://www.uniprot.org/uniprotkb/P02649/entry)  
   - No annotated isoforms (alternative splicing variants) are reported in the reviewed entry.

2. **Functional sequence intervals**  
   • Signal peptide — positions 1–18 — cleaved during maturation.  
   • LDL and other lipoprotein receptors binding — positions 158–168 — mediates interaction with LDLR, LRP1, LRP2, LRP8, and VLDLR.  
   • Lipid-binding and lipoprotein association — positions 210–290 — essential for association with chylomicrons, VLDL, HDL, and other lipoproteins.  
   • Homooligomerization — positions 266–317 — required for tetramer formation.  
   • Specificity for association with VLDL — positions 278–290 — determines preferential binding to VLDL.  
   • Heparin-binding site — positions 162–165 — binds heparan sulfate proteoglycans on c

### Test

In [100]:
with ToolCollection.from_mcp(
    server_parameters=server,
    trust_remote_code=True,
    structured_output=False
) as tools:
    agent = ToolCallingAgent(
        tools=[*tools.tools],
        add_base_tools=False,
        model=model,
        max_steps=1,
    )
    agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
    # agent.prompt_templates["final_answer"] = FINAL_RESULT_PROMPT

    # try:
    out = agent.run(user_prompt)
        # print(out)
    # except Exception as e:
    #     print(f"ERROR: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Protein: OCT4                                                                                                   │
│                                                                                                                 │
│ Retrieve all available UniProt data per the scope above, using only MCP tool outputs.                           │
│ Do not invent or infer missing information.                                                                     │
│ Search recommendations: Use more than 1 search results. String indices must be integers, not 'str'.             │
│ Rules and priorities:                                                                                           │
│ 1. **Always prefer reviewed (Swiss-Prot) entries** over unreviewed (TrEMBL) ones when both exist.               │
│ 2. If the search results include multiple entries, select the one explicitly marked as “UniProtKB reviewed      │
│ (Swiss-Prot)” — this is the canonical human record.                                                             │
│ 3. Never claim that data is “missing” if a reviewed canonical entry was found.                                  │
│ 4. If both a fragment and a full canonical entry are present, use the full reviewed one and ignore the          │
│ fragment.                                                                                                       │
│ 5. Use only the information retrieved from MCP tool outputs. Do not add facts from external knowledge.          │
│ 6. If no reviewed entry is found at all, then and only then mention that data may be incomplete.                │
│                                                                                                                 │
│ When writing the summary:                                                                                       │
│ - Use bullet points or short factual sentences under each section.                                              │
│ - Include FASTA links and UniProt IDs where available.                                                          │
│ - Do not include commentary about the search process or tool execution.                                         │
│ - Do not add speculative language like “may be”, “possibly”, “appears to”.                                      │
│ - All information must be based ONLY on the data retrieved by via tools.                                        │
│ If any of the sections are unavailable print "Info unavailable" - do not invent facts.                          │
│                                                                                                                 │
│ Write a concise, readable plain-text summary of the retrieved UniProt data.                                     │
│ Stick to this structure in output:                                                                              │
│ 1. **Canonical sequence and isoforms**                                                                          │
│    - Retrieve the canonical FASTA sequence and all annotated isoforms (alternative splicing variants).          │
│    - Note sequence length and UniProt accession for each.                                                       │
│                                                                                                                 │
│ 2. **Functional sequence intervals**                                                                            │
│    - From `features`: list all annotated *regions, domains, motifs, sites,* and *mutagenesis* entries.          │
│    - For each, include:                                                                                         │
│      • name of interval                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_by_gene' with arguments: {'gene': 'OCT4', 'organism': 'Homo sapiens', 'size': 5}          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "results": |
    {
      "entryType": "UniProtKB reviewed (Swiss-Prot)",
      "primaryAccession": "Q01860",
      "secondaryAccessions": |
        "A6NCS1",
        "A6NLL8",
        "D2IYK4",
        "P31359",
        "Q15167",
        "Q15168",
        "Q16422",
        "Q5STF3",
        "Q5STF4"
      ],
      "uniProtkbId": "PO5F1_HUMAN",
      "entryAudit": {
        "firstPublicDate": "1993-07-01",
        "lastAnnotationUpdateDate": "2025-06-18",
        "lastSequenceUpdateDate": "1993-07-01",
        "entryVersion": 235,
        "sequenceVersion": 1
      },
      "annotationScore": 5,
      "organism": {
        "scientificName": "Homo sapiens",
        "commonName": "Human",
        "taxonId": 9606,
        "lineage": |
          "Eukaryota",
          "Metazoa",
          "Chordata",
          "Craniata",
          "Vertebrata",
          "Euteleostomi",
          "Mammalia",
          "Eutheria",
          "Euarchontoglires",
          "Primates",
          "Haplorrhini",
          "Catarrhini",
          "Hominidae",
          "Homo"
        ]
      },
      "proteinExistence": "1: Evidence at protein level",
      "proteinDescription": {
        "recommendedName": {
          "fullName": {
            "value": "POU domain, class 5, transcription factor 1"
          }
        },
        "alternativeNames": |
          {
            "fullName": {
              "value": "Octamer-binding protein 3"
            },
            "shortNames": |
              {
                "value": "Oct-3"
              }
            ]
          },
          {
            "fullName": {
              "value": "Octamer-binding protein 4"
            },
            "shortNames": |
              {
                "value": "Oct-4"
              }
            ]
          },
          {
            "fullName": {
              "value": "Octamer-binding transcription factor 3"
            },
            "shortNames": |
              {
                "value": "OTF-3"
              }
            ]
          }
        ]
      },
      "genes": |
        {
          "geneName": {
            "value": "POU5F1"
          },
          "synonyms": |
            {
              "value": "OCT3"
            },
            {
              "value": "OCT4"
            },
            {
              "value": "OTF3"
            }
          ]
        }
      ],
      "comments": |
        {
          "texts": |
            {
              "evidences": |
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "18035408"
                }
              ],
              "value": "Transcription factor that binds to the octamer motif (5'-ATTTGCAT-3'). Forms a trimeric 
complex with SOX2 or SOX15 on DNA and controls the expression of a number of genes involved in embryonic 
development such as YES1, FGF4, UTF1 and ZFP206. Critical for early embryogenesis and for embryonic stem cell 
pluripotency"
            }
          ],
          "commentType": "FUNCTION"
        },
        {
          "texts": |
            {
              "evidences": |
                {
                  "evidenceCode": "ECO:0000250",
                  "source": "UniProtKB",
                  "id": "P20263"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "18191611"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "19274063"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "26687479"
                }
              ],
              "value": "Interacts with PKM. Interacts with WWP2. Interacts with UBE2I and ZSCAN10 (By similarity). 
Interacts with PCGF1 (PubMed:26687479). Interacts with ESRRB; recruits ESRRB nea

[Step 1: Duration 3.13 seconds| Input tokens: 4,275 | Output tokens: 38]

Reached max steps.

[Step 2: Duration 19.19 seconds| Input tokens: 39,346 | Output tokens: 1,130]

In [116]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [119]:
from uniprot import run_query

In [122]:
out = run_query('GHR')

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Protein: GHR                                                                                                    │
│                                                                                                                 │
│ Retrieve all available UniProt data per the scope above, using only MCP tool outputs.                           │
│ Do not invent or infer missing information.                                                                     │
│ Search recommendations: Use more than 1 search results. String indices must be integers, not 'str'.             │
│ Rules and priorities:                                                                                           │
│ 1. **Always prefer reviewed (Swiss-Prot) entries** over unreviewed (TrEMBL) ones when both exist.               │
│ 2. If the search results include multiple entries, select the one explicitly marked as “UniProtKB reviewed      │
│ (Swiss-Prot)” — this is the canonical human record.                                                             │
│ 3. Never claim that data is “missing” if a reviewed canonical entry was found.                                  │
│ 4. If both a fragment and a full canonical entry are present, use the full reviewed one and ignore the          │
│ fragment.                                                                                                       │
│ 5. Use only the information retrieved from MCP tool outputs. Do not add facts from external knowledge.          │
│ 6. If no reviewed entry is found at all, then and only then mention that data may be incomplete.                │
│                                                                                                                 │
│ When writing the summary:                                                                                       │
│ - Use bullet points or short factual sentences under each section.                                              │
│ - Include FASTA links and UniProt IDs where available.                                                          │
│ - Do not include commentary about the search process or tool execution.                                         │
│ - Do not add speculative language like “may be”, “possibly”, “appears to”.                                      │
│ - All information must be based ONLY on the data retrieved by via tools.                                        │
│ If any of the sections are unavailable print "Info unavailable" - do not invent facts.                          │
│                                                                                                                 │
│ Write a concise, readable plain-text summary of the retrieved UniProt data.                                     │
│ Stick to this structure in output:                                                                              │
│ 1. **Canonical sequence and isoforms**                                                                          │
│    - Retrieve the canonical FASTA sequence and all annotated isoforms (alternative splicing variants).          │
│    - Note sequence length and UniProt accession for each.                                                       │
│                                                                                                                 │
│ 2. **Functional sequence intervals**                                                                            │
│    - From `features`: list all annotated *regions, domains, motifs, sites,* and *mutagenesis* entries.          │
│    - For each, include:                                                                                         │
│      • name of interval                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_by_gene' with arguments: {'gene': 'GHR', 'organism': 'Homo sapiens', 'size': 5}           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "results": |
    {
      "entryType": "UniProtKB reviewed (Swiss-Prot)",
      "primaryAccession": "P10912",
      "secondaryAccessions": |
        "Q9HCX2"
      ],
      "uniProtkbId": "GHR_HUMAN",
      "entryAudit": {
        "firstPublicDate": "1989-07-01",
        "lastAnnotationUpdateDate": "2025-06-18",
        "lastSequenceUpdateDate": "1989-07-01",
        "entryVersion": 252,
        "sequenceVersion": 1
      },
      "annotationScore": 5,
      "organism": {
        "scientificName": "Homo sapiens",
        "commonName": "Human",
        "taxonId": 9606,
        "lineage": |
          "Eukaryota",
          "Metazoa",
          "Chordata",
          "Craniata",
          "Vertebrata",
          "Euteleostomi",
          "Mammalia",
          "Eutheria",
          "Euarchontoglires",
          "Primates",
          "Haplorrhini",
          "Catarrhini",
          "Hominidae",
          "Homo"
        ]
      },
      "proteinExistence": "1: Evidence at protein level",
      "proteinDescription": {
        "recommendedName": {
          "fullName": {
            "evidences": |
              {
                "evidenceCode": "ECO:0000303",
                "source": "PubMed",
                "id": "2825030"
              }
            ],
            "value": "Growth hormone receptor"
          },
          "shortNames": |
            {
              "value": "GH receptor"
            }
          ]
        },
        "alternativeNames": |
          {
            "fullName": {
              "value": "Somatotropin receptor"
            }
          }
        ],
        "contains": |
          {
            "recommendedName": {
              "fullName": {
                "evidences": |
                  {
                    "evidenceCode": "ECO:0000303",
                    "source": "PubMed",
                    "id": "2825030"
                  }
                ],
                "value": "Growth hormone-binding protein"
              },
              "shortNames": |
                {
                  "value": "GH-binding protein"
                },
                {
                  "value": "GHBP"
                }
              ]
            },
            "alternativeNames": |
              {
                "fullName": {
                  "value": "Serum-binding protein"
                }
              }
            ]
          }
        ],
        "flag": "Precursor"
      },
      "genes": |
        {
          "geneName": {
            "value": "GHR"
          }
        }
      ],
      "comments": |
        {
          "texts": |
            {
              "evidences": |
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "1549776"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "15690087"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "2825030"
                },
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "8943276"
                }
              ],
              "value": "Receptor for pituitary gland growth hormone (GH1) involved in regulating postnatal body 
growth (PubMed:1549776, PubMed:2825030, PubMed:8943276). On ligand binding, couples to the JAK2/STAT5 pathway 
(PubMed:1549776, PubMed:15690087, PubMed:2825030, PubMed:8943276)"
            }
          ],
          "commentType": "FUNCTION"
        },
        {
          "texts": |
            {
              "evidences": |
                {
                  "evidenceCode": "ECO:0000269",
                  "source": "PubMed",
                  "id": "2825030"
                }
              ],
              "value": "The soluble form (GHBP) acts as a reservoir of growth horm

[Step 1: Duration 4.54 seconds| Input tokens: 4,275 | Output tokens: 37]

Reached max steps.

[Step 2: Duration 30.08 seconds| Input tokens: 78,280 | Output tokens: 1,251]

In [123]:
print(out)

1. **Canonical sequence and isoforms**  
   - Canonical sequence: UniProtKB accession P10912, length 638 amino acids.  
     FASTA: [https://www.uniprot.org/uniprotkb/P10912/entry](https://www.uniprot.org/uniprotkb/P10912/entry)  
   - Isoform 1 (GHRfl): Accession P10912-1, length 638.  
   - Isoform 2 (GHRtr, GHR1-279): Accession P10912-2, length 295.  
   - Isoform 3 (GHR1-277): Accession P10912-3, length 294.  
   - Isoform 4 (GHRd3): Accession P10912-4, length 638.

2. **Functional sequence intervals**  
   - Signal peptide: 1–18.  
   - Extracellular domain: 19–264; contains ligand-binding region.  
   - Transmembrane domain: 265–288; helical.  
   - Cytoplasmic domain: 289–638.  
   - Fibronectin type-III domain: 151–254; structural domain in extracellular region.  
   - WSXWS motif: 240–244; required for proper protein folding and cell-surface receptor binding.  
   - Box 1 motif: 297–305; required for JAK2 interaction and activation.  
   - UbE motif: 340–349; required for ubiq